<a href="https://colab.research.google.com/github/aritrasinha531-ai/silver-invention/blob/main/code_for_ANN_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
uploaded

In [ ]:
df=pd.read_csv("DateFruit_Dataset.csv")
df

In [ ]:
x=df.drop("Class",axis=1)
y=df["Class"]

In [ ]:
df["Class"].unique()

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [ ]:
# ANN model
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader
x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)
x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)

In [ ]:
train_dataset=TensorDataset(x_train_tensor,y_train_tensor)
test_dataset=TensorDataset(x_test_tensor,y_test_tensor)
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [ ]:
class ANN(nn.Module):
  def __init__(self):
    super(ANN,self). __init__()
    self.model=nn.Sequential(
        # 1st hidden layer
        nn.Linear(x_train.shape[1],64),
        nn.ReLU(),
        # 2nd Hidden layer
        nn.Linear(64,64),
        nn.ReLU(),
        #output
        nn.Linear(64,7)
    )

  def forward(self,x):
    return self.model(x)

In [ ]:
model=ANN()
# loss and optim
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [ ]:
epochs=100
for epoch in range(epochs):
  model.train()

  running_loss=0.0
  for xb,yb in train_loader:
    optimizer.zero_grad()

    outputs=model(xb)
    loss=criteria(outputs,yb.squeeze().long())
    loss.backward()
    optimizer.step()
    running_loss+=loss.item()
  train_loss=running_loss/len(train_loader)
  print(f"epoch {epoch+1}/{epochs},loss:{train_loss}")

In [ ]:
# evaluate
model.eval()
total=0
correct=0
with torch.no_grad():
  for xb,yb in test_loader:
    outputs=model(xb)
    _,predicted=torch.max(outputs.data,1)
    total+=yb.size(0)
    correct+=(predicted==yb).sum().item()
accuracy=correct/total
print(f"accuracy:{accuracy}")


In [ ]:
print("end of project")